In [ ]:
pip install requests

12

In [3]:
import requests
import pandas as pd

# Function to fetch product data from OpenFoodFacts API
def get_openfoodfacts_product(product_name):
    url = "https://world.openfoodfacts.org/cgi/search.pl"
    params = {
        "action": "process",
        "json": "true",
        "search_terms": product_name,
        "page_size": 1
    }
    response = requests.get(url, params=params)
    if response.status_code == 200:
        data = response.json()
        if data['products']:
            return data['products'][0]  # Return the first matching product
    return None

# Function to estimate CO2 impact from Eco-Score
def calculate_co2_from_ecoscore(ecoscore):
    # Mapping Eco-Score grades to estimated CO2 emissions (kg CO2e per kg of product)
    ecoscore_co2_mapping = {
        "a": 0.5,  # Low impact
        "b": 1.0,
        "c": 1.5,
        "d": 2.5,
        "e": 3.5  # High impact
    }
    return ecoscore_co2_mapping.get(ecoscore.lower(), 3.0)  # Default to higher impact if unknown

# Function to process a shopping list
def process_shopping_list(shopping_list):
    results = []
    total_co2 = 0

    for product_name in shopping_list:
        product_data = get_openfoodfacts_product(product_name)
        if product_data:
            ecoscore = product_data.get("ecoscore_grade", "n/a")
            co2_impact = calculate_co2_from_ecoscore(ecoscore) if ecoscore != "n/a" else "Unknown"
            total_co2 += co2_impact if isinstance(co2_impact, float) else 0
            
            results.append({
                "Product": product_name,
                "Eco-Score": ecoscore.upper() if ecoscore != "n/a" else "Unknown",
                "Estimated CO2 Impact (kg CO2e)": co2_impact
            })
        else:
            results.append({
                "Product": product_name,
                "Eco-Score": "Not Found",
                "Estimated CO2 Impact (kg CO2e)": "Unknown"
            })

    return results, total_co2

# Main program
if __name__ == "__main__":
    # Example shopping list
    shopping_list = ["Oreo cookies", "Milk", "Bananas", "Beef steak", "Rice"]

    # Process the shopping list
    results, total_co2 = process_shopping_list(shopping_list)

    # Display results
    results_df = pd.DataFrame(results)
    print("Shopping List Analysis:")
    print(results_df)

    print(f"\nTotal Estimated CO2 Impact for the Shopping List: {total_co2:.2f} kg CO2e")

Shopping List Analysis:
        Product Eco-Score  Estimated CO2 Impact (kg CO2e)
0  Oreo cookies         E                             3.5
1          Milk         B                             1.0
2       Bananas         A                             0.5
3    Beef steak         F                             3.0
4          Rice         B                             1.0

Total Estimated CO2 Impact for the Shopping List: 9.00 kg CO2e
